# 00. 환경·데이터·KoBART 생성 확인

수업 `7_1.ipynb`의 BART 사용 흐름으로 한국어 뉴스 한 건의 생성을 확인합니다.

커널은 **Python 3.11 (Team4 News GPU)**를 선택하고 위에서부터 Shift+Enter로 실행하세요. 각 코드 셀은 앞 셀의 변수를 사용합니다. 처음 모델을 내려받을 때는 시간이 걸립니다.

이번 단계는 실행 점검입니다. 파인튜닝과 test 성능 평가는 수행하지 않습니다.

## 1. 프로젝트 경로와 실행 환경

프로젝트 가상환경을 선택했는지 확인하고 다운로드 캐시를 프로젝트 폴더에 보관합니다.

In [1]:
import os
import sys
from pathlib import Path

PROJECT = Path("C:/Users/KDS-30/Desktop/Projects/06_Team_Project4")
assert PROJECT.exists(), "프로젝트 경로를 확인하세요."
assert Path(sys.prefix).resolve() == (PROJECT / ".venv").resolve(), "커널을 Python 3.11 (Team4 News GPU)로 바꾸세요."

os.environ["HF_HOME"] = str(PROJECT / ".cache" / "huggingface")
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
RESULTS = PROJECT / "results" / "00_environment_data_kobart"
RESULTS.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version)
print("실행 파일:", sys.executable)
print("결과 폴더:", RESULTS)

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
실행 파일: c:\Users\KDS-30\Desktop\Projects\06_Team_Project4\.venv\Scripts\python.exe
결과 폴더: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\results\00_smoke_check


## 2. 라이브러리와 GPU 확인

CUDA 사용 여부와 실제 GPU 행렬 연산을 확인합니다. GPU 이름만 표시되는 것보다 확실한 점검입니다.

In [2]:
import json
import time
import platform
import importlib.metadata
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import AutoTokenizer, BartForConditionalGeneration, set_seed

set_seed(42)
assert torch.cuda.is_available(), "CUDA를 사용할 수 없습니다."
device = torch.device("cuda")
x = torch.randn(128, 128, device=device)
y = x @ x.T
torch.cuda.synchronize()
assert torch.isfinite(y).all().item()
environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "vram_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
    "cuda_matmul_passed": True,
    "packages": {p: importlib.metadata.version(p) for p in
                 ["transformers", "datasets", "tokenizers", "huggingface-hub"]},
}
display(pd.Series(environment, name="환경"))
del x, y

python                                                           3.11.9
torch                                                       2.7.1+cu128
cuda                                                               12.8
gpu                                  NVIDIA GeForce RTX 5060 Laptop GPU
vram_gib                                                           7.96
cuda_matmul_passed                                                 True
packages              {'transformers': '4.44.2', 'datasets': '2.21.0...
Name: 환경, dtype: object

## 3. 데이터 불러오기

입력은 `document`, 정답 요약은 `summary`입니다. 제공된 분할을 유지하고, 이번 점검에서는 train 샘플만 읽습니다. 재현을 위해 다운로드한 버전도 기록합니다.

In [3]:
DATASET_ID = "daekeun-ml/naver-news-summarization-ko"
MODEL_ID = "gogamza/kobart-base-v2"
api = HfApi()
dataset_revision = api.dataset_info(DATASET_ID).sha
model_revision = api.model_info(MODEL_ID).sha
ds = load_dataset(DATASET_ID, revision=dataset_revision)
split_counts = {name: len(data) for name, data in ds.items()}
display(pd.DataFrame(list(split_counts.items()), columns=["split", "건수"]))
print("컬럼:", ds["train"].column_names)
print("데이터 버전:", dataset_revision)

,split,건수
0,train,22194
1,validation,2466
2,test,2740


컬럼: ['date', 'category', 'press', 'title', 'document', 'link', 'summary']
데이터 버전: 916dfe6aed651b2ecd9d04fa0a704f8953e90762


Hugging Face에 있는 뉴스 데이터셋을 내려받아 노트북에서 사용할 수 있게 불러오는 과정
- date: 기사 날짜
- category: 경제 또는 IT과학
- press: 언론사
- title: 제목
- document: 기사 본문, 모델에 넣을 입력
- link: 원문 링크
- summary: 정답 요약, 모델이 학습할 목표

제작자가 미리 train, val, test로 나눠서 제공한것

## 4. 기사와 정답 요약 3건 읽기

제목, 원문, 정답을 함께 확인합니다. 정답에 반복이나 어색한 표현이 있는지도 살펴보세요.

In [4]:
samples = [ds["train"][i] for i in range(3)]
for i, row in enumerate(samples):
    assert isinstance(row["document"], str) and row["document"].strip()
    assert isinstance(row["summary"], str) and row["summary"].strip()
    print(f"샘플 {i} | {row['title']}")
    print("원문:", row["document"])
    print("정답 요약:", row["summary"])
    print()


샘플 0 | 추경호 중기 수출지원 총력 무역금융 40조 확대
원문: 앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.
정답 요약: 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, 정부가 하반기에 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 결정한 가운데, 특히 수출 중소기업의 물류난 해소를 위해 무역금융

train set에서 앞의 기사 3건을 꺼내서 그냥 보여준것

## 5. KoBART 토크나이저와 모델 로딩
- 한국어 문장을 숫자로 바꿔주는 토크나이저와, 그 숫자를 처리해 문장을 생성하는 KoBART 모델을 불러와 GPU에 올리는 단계
- 한국어 사전학습 모델을 사용합니다. 뉴스 요약용 파인튜닝 전이므로 출력 품질보다 생성이 정상 작동하는지 확인합니다.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=model_revision)
model = BartForConditionalGeneration.from_pretrained(
    MODEL_ID, revision=model_revision, use_safetensors=True
).to(device)
model.eval()
print("모델:", MODEL_ID)
print("모델 버전:", model_revision)
print("파라미터 수:", f"{sum(p.numel() for p in model.parameters()):,}")
print("모델 장치:", next(model.parameters()).device)
print("특수 토큰:", tokenizer.special_tokens_map)

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.
c:\Users\KDS-30\Desktop\Projects\06_Team_Project4\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


모델: gogamza/kobart-base-v2
모델 버전: f5899cc556216353e23ee36b66fad52b902066c5
파라미터 수: 123,859,968
모델 장치: cuda:0
특수 토큰: {'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'mask_token': '<mask>'}


## 6. 기사 한 건 토큰화

첫 train 기사를 최대 512토큰으로 변환합니다. 원래 토큰 수와 잘림 여부를 함께 기록합니다.

In [6]:
sample_index = 0
row = samples[sample_index]
full_input_tokens = len(tokenizer(row["document"])["input_ids"])
inputs = tokenizer(
    row["document"], return_tensors="pt", truncation=True, max_length=512,
    return_token_type_ids=False  # BART는 문장 구분용 token_type_ids를 사용하지 않습니다.
).to(device)
input_tokens = inputs["input_ids"].shape[1]
print("원래 토큰 수:", full_input_tokens)
print("모델에 들어가는 토큰 수:", input_tokens)
print("잘림 여부:", full_input_tokens > input_tokens)
print("input_ids 크기:", tuple(inputs["input_ids"].shape))
print("attention_mask 크기:", tuple(inputs["attention_mask"].shape))

원래 토큰 수: 315
모델에 들어가는 토큰 수: 315
잘림 여부: False
input_ids 크기: (1, 315)
attention_mask 크기: (1, 315)


ㄴ> 이 코드는 samples에서 문서 하나를 꺼내서, BART 토크나이저에 넣기 전 원래 토큰 수와 max_length=512 적용 후 실제 모델에 들어가는 토큰 수를 비교하는 코드

## 7. 실제 생성 실행

정답 요약을 전달하지 않고 기사만 전달합니다. 가장 높은 확률의 토큰을 선택하는 greedy 방식으로 최대 128개 새 토큰을 생성합니다.

In [7]:
generation_config = {"max_new_tokens": 128, "num_beams": 1, "do_sample": False}
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
started = time.perf_counter()
with torch.inference_mode():
    generated_ids = model.generate(**inputs, **generation_config)
torch.cuda.synchronize()
generation_seconds = time.perf_counter() - started
prediction = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
assert generated_ids.shape[1] > 1, "생성된 토큰이 없습니다."
print("생성 시간(초):", round(generation_seconds, 3))
print("최대 GPU 텐서 메모리(MiB):", round(torch.cuda.max_memory_allocated() / 2**20, 1))
print("생성 결과:", prediction)

생성 시간(초): 1.481
최대 GPU 텐서 메모리(MiB): 503.0
생성 결과: 가가 올해 상반기 상반기 올해 상반기 올해 상반기 우리나라 상반기 올해 상반기 우리나라 상반기 올해 상반기 우리나라 무역수지는 올해 상반기 올해 상반기 우리나라 무역수지는 역대 올해 올해 올해 상반기 올해 상반기 올해 상반기 우리나라 무역수지는 역대 최악인 103억 적자를 적자를 적자를 적자를 적자를 적자를을을을을을을을을을을 올해 상반기 올해 상반기 수출을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을을


## 8. 원문·정답·생성 결과 비교

파인튜닝 전 모델은 원문을 복사하거나 반복할 수 있습니다. 이 출력으로 요약 모델의 최종 성능을 판단하지 않습니다.

In [8]:
print("제목:", row["title"])
print("\n[원문]\n", row["document"])
print("\n[데이터셋 정답 요약]\n", row["summary"])
print("\n[파인튜닝 전 생성 결과]\n", prediction)

제목: 추경호 중기 수출지원 총력 무역금융 40조 확대

[원문]
 앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.

[데이터셋 정답 요약]
 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, 정부가 하반기에 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 결정한 가운데, 특히 수출 중소기업의 물류난 해소를

## 9. 점검 결과 저장

환경, 데이터 개수, 모델 버전, 생성 설정과 실제 결과를 JSON으로 저장합니다.

In [10]:
report = {
    "status": "passed",
    "environment": environment,
    "dataset": DATASET_ID,
    "dataset_revision": dataset_revision,
    "split_counts": split_counts,
    "columns": ds["train"].column_names,
    "model": MODEL_ID,
    "model_revision": model_revision,
    "sample_split": "train",
    "sample_index": sample_index,
    "full_input_tokens": full_input_tokens,
    "input_tokens": input_tokens,
    "truncated": full_input_tokens > input_tokens,
    "generation_config": generation_config,
    "generation_seconds": round(generation_seconds, 3),
    "peak_gpu_allocated_mib": round(torch.cuda.max_memory_allocated() / 2**20, 1),
    "prediction": prediction,
    "sample": row,
    "note": "파인튜닝 전 실행 점검. 학습 및 test 성능 평가 미실시.",
}
(RESULTS / "report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
)
(RESULTS / "train_samples.json").write_text(
    json.dumps(samples, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("점검 완료:", RESULTS / "report.json")

점검 완료: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\results\00_smoke_check\report.json


## 다음 단계

환경·데이터 로딩·GPU 생성이 확인되면 데이터 길이, 중복, 노이즈를 분석하고 소량 데이터로 파인튜닝을 시작합니다.

환경 설치 메모: 원본 requirements는 보존했습니다. Korpora의 의존성으로 설치되는 dataclasses 0.8은 Python 3.11과 버전 조건이 충돌하여 0.6으로 고정했습니다. 실제 Python 3.11 코드는 표준 라이브러리 dataclasses를 사용합니다.